In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict, load_from_disk
from rdkit import Chem

from transformers import (
    AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    AutoTokenizer, RobertaModel, RobertaTokenizer, RobertaTokenizerFast,
    RobertaConfig, RobertaPreTrainedModel, DebertaV2Model, DebertaV2Config,
    DebertaTokenizerFast, LongformerModel, LongformerConfig,
    LongformerForMaskedLM, LongformerTokenizerFast,
    AutoConfig, Trainer, TrainingArguments, pipeline,
    default_data_collator
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_pt_utils import get_parameter_names

# Tokenizers
from tokenizers import ByteLevelBPETokenizer
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

# Paths
CONVERTED_DIR   = "data/newTrainVal"           # output of convert_uniprot_to_sequence.py
SAVE_DIR        = "Data3/inputdataMMAT"        # where to save processed datasets
TRAIN_SAVE_PATH = os.path.join(SAVE_DIR, "train_datasets")
TEST_SAVE_PATH  = os.path.join(SAVE_DIR, "test_datasets")
TASKS_PATH      = os.path.join(SAVE_DIR, "tasks.json")

LABEL_COL       = "pchembl_value"             # regression target
SPLIT_SEED      = 42
TEST_SIZE       = 0.2

model_pth = "../Pre-Training/PubchemModelDeberta"
model_pth2 = "../Pre-Training/ProteinModelDeb"
tokenizer = DebertaTokenizerFast.from_pretrained(model_pth)
tokenizer2 = DebertaTokenizerFast.from_pretrained(model_pth2)

In [ ]:

def canonicalize_smiles(example):
    mol = Chem.MolFromSmiles(example["compound_iso_smiles"], sanitize=True)
    if mol is not None:
        example["compound_iso_smiles"] = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
        return example
    return None  

def tokenization3(example):
    batch_size = len(example["compound_iso_smiles"])  

    smiles_encodings = tokenizer(
        example["compound_iso_smiles"], padding="max_length", truncation=True, max_length=512
    )

    tokenized_full = tokenizer2(example["target_sequence"], return_offsets_mapping=True)

    input_ids_full_batch = tokenized_full["input_ids"]

    if isinstance(input_ids_full_batch[0], list):
        input_ids_full_batch = [tokens if isinstance(tokens, list) else [tokens] for tokens in input_ids_full_batch]

    input_ids2_batch, input_ids3_batch = [], []
    attention_mask2_batch, attention_mask3_batch = [], []

    for input_ids_full in input_ids_full_batch:
        input_ids_1 = input_ids_full[:512] 
        input_ids_2 = input_ids_full[512:]

        input1_len = len(input_ids_1)
        input2_len = min(len(input_ids_2), 512)  

        input_ids_1 = input_ids_1 + [tokenizer.pad_token_id] * (512 - input1_len)
        input_ids_2 = input_ids_2[:input2_len] + [tokenizer.pad_token_id] * (512 - input2_len)

        attention_mask2 = [1] * input1_len + [0] * (512 - input1_len)
        attention_mask3 = [1] * input2_len + [0] * (512 - input2_len)

        input_ids2_batch.append(input_ids_1)
        input_ids3_batch.append(input_ids_2)
        attention_mask2_batch.append(attention_mask2)
        attention_mask3_batch.append(attention_mask3)

    return {
        "input_ids": smiles_encodings["input_ids"],
        "attention_mask": smiles_encodings["attention_mask"],
        "input_ids2": input_ids2_batch,
        "input_ids3": input_ids3_batch,
        "attention_mask2": attention_mask2_batch,
        "attention_mask3": attention_mask3_batch,
    }
    

def process_and_split_file(csv_path: str, test_size: float = 0.2, seed: int = 42):
    """
    Load one converted CSV, canonicalize SMILES, tokenize, rename label,
    and return (train_dataset, test_dataset, task_name).
    """
    df = pd.read_csv(csv_path)
    for col in ["compound_iso_smiles", "target_sequence", LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {csv_path}")

    before = len(df)
    df = df.dropna(subset=["compound_iso_smiles", "target_sequence", LABEL_COL])
    dropped = before - len(df)
    if dropped:
        print(f"  Dropped {dropped} rows with NaN in key columns.")

    df = df.rename(columns={LABEL_COL: "labels"})

    df = df[["compound_iso_smiles", "target_sequence", "labels"]]
    
    train_idx = df.sample(frac=1 - test_size, random_state=seed).index
    df_train  = df.loc[train_idx].reset_index(drop=True)
    df_test   = df.drop(index=train_idx).reset_index(drop=True)
    print(f"  Split → train: {len(df_train)}  test: {len(df_test)}")

    results = {}
    for split_name, split_df in [("train", df_train), ("test", df_test)]:
        ds = Dataset.from_pandas(split_df)

        ds = ds.map(canonicalize_smiles).filter(lambda x: x is not None)
        ds = ds.map(tokenization3, batched=True)
        drop_cols = [c for c in ["compound_iso_smiles", "target_sequence",
                                  "__index_level_0__"] if c in ds.column_names]
        ds = ds.remove_columns(drop_cols)

        ds.set_format("torch")
        results[split_name] = ds

    task_name = Path(csv_path).stem.replace("_converted", "")

    return results["train"], results["test"], task_name


def build_mmat_datasets(converted_dir: str, test_size: float = 0.2, seed: int = 42):
    """
    Process all *_converted.csv files and return
        train_dict  {task_name: Dataset}
        test_dict   {task_name: Dataset}
        tasks_dict  {task_name: ["pchembl_value"]}
    """
    csv_files = sorted(Path(converted_dir).glob("*_converted.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No *_converted.csv files found in {converted_dir}")

    print(f"Found {len(csv_files)} file(s): {[f.name for f in csv_files]}\n")

    train_dict = {}
    test_dict  = {}
    tasks_dict = {}

    for csv_path in csv_files:
        print(f"Processing: {csv_path.name}")
        train_ds, test_ds, task_name = process_and_split_file(
            str(csv_path), test_size=test_size, seed=seed
        )
        train_dict[task_name] = train_ds
        test_dict[task_name]  = test_ds
        tasks_dict[task_name] = [LABEL_COL]   # regression, single label
        print(f"  ✓ {task_name}  train={len(train_ds)}  test={len(test_ds)}")
        print(f"    features: {train_ds.column_names}\n")

    return train_dict, test_dict, tasks_dict

In [ ]:
mmat_exists = (
    os.path.exists(TRAIN_SAVE_PATH) and
    os.path.exists(TEST_SAVE_PATH)  and
    os.path.exists(TASKS_PATH)
)

if mmat_exists:
    print("MMAT datasets already saved — loading from disk.")
    mmat_train = dict(load_from_disk(TRAIN_SAVE_PATH))
    mmat_test  = dict(load_from_disk(TEST_SAVE_PATH))
    with open(TASKS_PATH) as f:
        mmat_tasks = json.load(f)
else:
    mmat_train, mmat_test, mmat_tasks = build_mmat_datasets(
        CONVERTED_DIR, test_size=TEST_SIZE, seed=SPLIT_SEED
    )

    os.makedirs(SAVE_DIR, exist_ok=True)
    DatasetDict(mmat_train).save_to_disk(TRAIN_SAVE_PATH)
    DatasetDict(mmat_test).save_to_disk(TEST_SAVE_PATH)
    with open(TASKS_PATH, "w") as f:
        json.dump(mmat_tasks, f)
    print(f"Saved to {SAVE_DIR}")

In [ ]:
def get_notebook_name():
    try:
        return ipynbname.name()
    except Exception:
        return "unknown_notebook"

notebook_name = get_notebook_name().split(".")[0]  # Remove file extension if any
os.makedirs("result_graphs", exist_ok=True)

In [ ]:
save_dir = "Data3"
inputdata_dir = os.path.join(save_dir, "inputdataAbsolute")

train_datasets_path = os.path.join(inputdata_dir, "train_datasets")
test_datasets_path = os.path.join(inputdata_dir, "test_datasets")
tasks_path = os.path.join(inputdata_dir, "tasks.json")

if os.path.exists(train_datasets_path) and os.path.exists(test_datasets_path):
    train_datasets = load_from_disk(train_datasets_path)
    test_datasets = load_from_disk(test_datasets_path)
    with open(tasks_path, "r") as f:
        tasks = json.load(f)
    updated_datasets = {"train": train_datasets, "test": test_datasets}
    datasets_exist = True
    print("Datasets and tasks successfully loaded.")
else:
    datasets_exist = False
    print("Datasets not found. Skipping loading.")

In [ ]:

Datasets_Available = {
    "bace": {"problem_type": "multi_label_classification", "num_labels": 1},
    "hiv": {"problem_type": "multi_label_classification", "num_labels": 1},
    "bbbp": {"problem_type": "multi_label_classification", "num_labels": 1},
    "clintox": {"problem_type": "multi_label_classification", "num_labels": 2},
    "tox21": {"problem_type": "multi_label_classification","num_labels": 12},
    "muv": {"problem_type": "multi_label_classification", "num_labels": 17},
    "sider": {"problem_type": "multi_label_classification", "num_labels": 27},
    "toxcast": {"problem_type": "multi_label_classification", "num_labels": 617},
    "delaney": { "problem_type": "regression", "num_labels": 1},
    "freesolv": { "problem_type": "regression", "num_labels": 1},
    "lipo": {"problem_type": "regression", "num_labels": 1},
}

In [ ]:
for task_name, ds in mmat_train.items():
    updated_datasets["train"][task_name] = ds

for task_name, ds in mmat_test.items():
    updated_datasets["test"][task_name] = ds

print("\nupdated_datasets['train'] keys:", list(updated_datasets["train"].keys()))
print("\nSpot-check features per dataset type:")
for name, ds in updated_datasets["train"].items():
    print(f"  {name:30s}  rows={len(ds):6d}  cols={ds.column_names}")

for task_name in mmat_tasks:
    Datasets_Available[task_name] = {
        "problem_type": "regression",
        "num_labels": 1,
        "used_encoders": "multiencoder"
    }

num_labels_list   = [v["num_labels"]    for v in Datasets_Available.values()]
problem_type_list = [v["problem_type"]  for v in Datasets_Available.values()]

print("\nDatasets_Available keys:", list(Datasets_Available.keys()))
print("num_labels_list:", num_labels_list)